In [19]:
import torch
from collections import OrderedDict  #这个类是字典dict的一个子类，用于创建有序的字典。普通字典中元素顺序是无序的，在OrderedDict中元素的顺序是有序的，元素的顺序是按照它们被添加到字典中的顺序决定的。
from pyDOE import lhs 
import numpy as np
import matplotlib.pyplot as plt
import scipy.io #导入了scipy库中的io模块。scipy.io模块包含了一些用于文件输入/输出的函数，例如读取和写入.mat文件（MATLAB格式）
from scipy.interpolate import griddata #`scipy.interpolate`是`scipy`库中的一个模块，提供了许多插值工具，用于在给定的离散数据点之间进行插值和拟合。`griddata`是这个模块中的一个函数，用于在无规则的数据点上进行插值。
import random
import skopt #用于优化问题的库，特别是机器学习中的超参数优化
from distutils.version import LooseVersion #distutils是Python的一个标准库，用于构建和安装Python包。LooseVersion是一个类，用于比较版本号
from mpl_toolkits.axes_grid1 import make_axes_locatable #`mpl_toolkits.axes_grid1`是`matplotlib`库的一个模块，提供了一些高级的工具来控制matplotlib图形中的坐标轴和颜色条。`make_axes_locatable`是模块中的一个函数，用于创建一个可分割的坐标轴。可以在这个坐标轴的四个方向（上、下、左、右）添加新的坐标轴或颜色条。
import matplotlib.gridspec as gridspec #是`matplotlib`库的一个模块，用于创建一个网格布局来放置子图。在`matplotlib`中可以创建一个或多个子图（subplot），每个子图都有自己的坐标轴，并可以在其中绘制图形。`gridspec`模块提供了一个灵活的方式来创建和放置子图。
import time #一个内置模块，用于处理时间相关的操作。
from tqdm import tqdm #一个快速，可扩展的python进度条库，可以在python长循环中添加一个进度提示信息，用户只需要封装任意的迭代器tqdm(iterator)。
import os
import pickle
import timeit #用于计时和测量小段代码的执行时间
import seaborn as sns  # 导入seaborn库用于绘制密度图
import pandas as pd #一个用于数据操作和分析的库，提供了数据结构和数据分析工具，特别是用于处理表格数据（类似于Excel中的数据表）
import sys #导入sys模块。sys模块提供了一些变量和函数，用于与 Python解释器进行交互和访问。例如，sys.path 是一个 Python 在导入模块时会查找的路径列表，sys.argv 是一个包含命令行参数的列表，sys.exit() 函数可以用于退出 Python 程序。导入 sys 模块后，你就可以在你的程序中使用这些变量和函数了。
sys.path.insert(0, '../../..') #在 Python的sys.path列表中插入一个新的路径。sys.path是一个 Python 在导入模块时会查找的路径列表。新的路径'../../Utilities/'相对于当前脚本的路径。当你尝试导入一个模块时，Python 会在 sys.path 列表中的路径下查找这个模块。通过在列表开始位置插入一个路径，你可以让 Python 优先在这个路径下查找模块。这在你需要导入自定义模块或者不在 Python 标准库中的模块时非常有用。
from util import *
sys.path.insert(0, '../') #在 Python的sys.path列表中插入一个新的路径。sys.path是一个 Python 在导入模块时会查找的路径列表。新的路径'../../Utilities/'相对于当前脚本的路径。当你尝试导入一个模块时，Python 会在 sys.path 列表中的路径下查找这个模块。通过在列表开始位置插入一个路径，你可以让 Python 优先在这个路径下查找模块。这在你需要导入自定义模块或者不在 Python 标准库中的模块时非常有用。
from KAN import ModelKAN
from Transformer import ModelFormer
from DNN import ModelDNN

In [20]:
class ExperimentConfig:
    def __init__(self, model_type='DNN', train_method='ActiveLearning', 
                 opt_list='Adam+LBFGS', use_pretrained=False):
        self.model_type = model_type           # 'DNN', 'KAN', 'Transformer'
        self.train_method = train_method       # 'Standard', 'Residual', 'Chaos', 'Failure', 'Failure + Residual'
        self.opt_list = opt_list               # 'LBFGS', 'Adam+LBFGS'
        self.use_pretrained = use_pretrained   # True, False
        # 物理与采样参数
        self.N_f = 51 * 51
        self.N_f_pretrain = self.N_f // 2
        self.nIter = 10000 if opt_list == 'Adam+LBFGS' else 0 #Adam训练次数，使用两个优化器时，Adam训练10000次，只使用LBFGS时，Adam训练0次
        self.nIterLBFGS = 500 if opt_list == 'Adam+LBFGS' else 1000 #LBFGS训练次数，使用两个优化器时，LBFGS在Adam基础上训练500次，只使用LBFGS时，LBFGS训练1000次
        self.nIter_pretrain = 1000 if opt_list == 'Adam+LBFGS' else 0 #预训练Adam训练次数，使用两个优化器时，预训练Adam训练1000次，只使用LBFGS时，预训练Adam训练0次
        self.nIterLBFGS_pretrain = 0 if opt_list == 'Adam+LBFGS' else 500 #预训练LBFGS训练次数，使用两个优化器时，预训练LBFGS训练0次，只使用LBFGS时，预训练LBFGS训练500次
        self.nIterLBFGS = 600 if train_method in ['Failure + Residual', 'Failure + Residual + Chaos', 'Failure', 'Residual', 'Chaos'] and opt_list == 'LBFGS' else self.nIterLBFGS 
model_list = ['DNN', 'Transformer', 'KAN']
method_list = ['Standard', 'Residual', 'Chaos', 'Failure', 'Failure + Residual', 'Failure + Residual + Chaos']
opt_list = ['Adam+LBFGS', 'LBFGS']
config = ExperimentConfig(
                model_type='KAN', 
                train_method='Standard', 
                opt_list='LBFGS',
                use_pretrained=True
            )

In [21]:
torch.cuda.set_device(1) #设置当前使用的GPU设备。这里设置为1号GPU设备（第二块显卡）。

# CUDA support 

#设置pytorch的设备，代表了在哪里执行张量积算，设备可以是cpu或者cuda（gpu），并将这个做运算的设备对象存储在变量device中，后续张量计算回在这个设备上执行
if torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')

In [22]:
# the physics-guided neural network
class PhysicsInformedNN():
    # Initialize the class
    def __init__(self, config, b_left, b_right, b_upper, b_lower, X_f, X_f_pretrain): #这个类包含的第一个方法__init__，这是一个特殊的方法，也就是这个类的构造函数，用于初始化新创建的对象，接受了几个参数

        self.config = config

        
        # --- 模型选择 ---
        if config.model_type == 'DNN':
            self.dnn = ModelDNN([2, 512, 512, 512, 512, 2, 1]).to(device)
            # self.dnn.apply(self.init_weights) 
            self.optimizer_LBFGS = torch.optim.LBFGS(self.dnn.parameters(), line_search_fn='strong_wolfe')
            self.optimizer_LBFGS_pretrain = torch.optim.LBFGS(self.dnn.parameters(), line_search_fn='strong_wolfe')
        elif config.model_type == 'Transformer':
            self.dnn = ModelFormer(d_out=1, d_hidden=512, d_model=32, N=1, heads=2).to(device)
            # self.dnn.apply(self.init_weights) 
            self.optimizer_LBFGS = torch.optim.LBFGS(self.dnn.parameters(), line_search_fn='strong_wolfe') 
            self.optimizer_LBFGS_pretrain = torch.optim.LBFGS(self.dnn.parameters(), line_search_fn='strong_wolfe') 
        elif config.model_type == 'KAN':
            self.dnn = ModelKAN(width=[2, 5, 5, 1], grid=5, k=3, grid_eps=1.0, \
                                  noise_scale_base=0.25, device=device).to(device)
            self.optimizer_LBFGS = torch.optim.LBFGS(self.dnn.parameters(), line_search_fn='strong_wolfe', tolerance_grad = 1e-8, tolerance_change= 1e-10)
            self.optimizer_LBFGS_pretrain = torch.optim.LBFGS(self.dnn.parameters(), line_search_fn='strong_wolfe', tolerance_grad = 1e-8, tolerance_change= 1e-10)


        # --- 优化器初始化 ---
        self.optimizer_Adam = torch.optim.Adam(self.dnn.parameters())
        self.optimizer_Adam_pretrain = torch.optim.Adam(self.dnn.parameters())
        

        if config.model_type == 'Transformer':
            X_f_pretrain = make_time_sequence(X_f_pretrain, num_step=5, step=1e-4) 
            X_f = make_time_sequence(X_f, num_step=5, step=1e-4) 
            b_left = make_time_sequence(b_left, num_step=5, step=1e-4) 
            b_right = make_time_sequence(b_right, num_step=5, step=1e-4) 
            b_upper = make_time_sequence(b_upper, num_step=5, step=1e-4) 
            b_lower = make_time_sequence(b_lower, num_step=5, step=1e-4) 
            
            self.x_f = torch.tensor(X_f[:,:, 0:1], requires_grad=True).float().to(device)
            self.t_f = torch.tensor(X_f[:,:, 1:2], requires_grad=True).float().to(device)
            self.x_f_pretrain = torch.tensor(X_f_pretrain[:,:, 0:1], requires_grad=True).float().to(device)
            self.t_f_pretrain = torch.tensor(X_f_pretrain[:,:, 1:2], requires_grad=True).float().to(device)

            self.x_left = torch.tensor(b_left[:,:, 0:1], requires_grad=True).float().to(device) 
            self.t_left = torch.tensor(b_left[:,:, 1:2], requires_grad=True).float().to(device)
            self.x_right = torch.tensor(b_right[:,:, 0:1], requires_grad=True).float().to(device) 
            self.t_right = torch.tensor(b_right[:,:, 1:2], requires_grad=True).float().to(device)
            self.x_upper = torch.tensor(b_upper[:,:, 0:1], requires_grad=True).float().to(device) 
            self.t_upper = torch.tensor(b_upper[:,:, 1:2], requires_grad=True).float().to(device)
            self.x_lower = torch.tensor(b_lower[:,:, 0:1], requires_grad=True).float().to(device)
            self.t_lower = torch.tensor(b_lower[:,:, 1:2], requires_grad=True).float().to(device)
            
        else:
            self.x_f = torch.tensor(X_f[:, 0:1], requires_grad=True).float().to(device)
            self.t_f = torch.tensor(X_f[:, 1:2], requires_grad=True).float().to(device)
            self.x_f_pretrain = torch.tensor(X_f_pretrain[:, 0:1], requires_grad=True).float().to(device)
            self.t_f_pretrain = torch.tensor(X_f_pretrain[:, 1:2], requires_grad=True).float().to(device)

            self.x_left = torch.tensor(b_left[:, 0:1], requires_grad=True).float().to(device) 
            self.t_left = torch.tensor(b_left[:, 1:2], requires_grad=True).float().to(device)
            self.x_right = torch.tensor(b_right[:, 0:1], requires_grad=True).float().to(device) 
            self.t_right = torch.tensor(b_right[:, 1:2], requires_grad=True).float().to(device)
            self.x_upper = torch.tensor(b_upper[:, 0:1], requires_grad=True).float().to(device) 
            self.t_upper = torch.tensor(b_upper[:, 1:2], requires_grad=True).float().to(device)
            self.x_lower = torch.tensor(b_lower[:, 0:1], requires_grad=True).float().to(device)
            self.t_lower = torch.tensor(b_lower[:, 1:2], requires_grad=True).float().to(device)


        self.iter = 0 #记录迭代次数 

        self.loss_value = [] #创建一个空列表，用于存储损失值

    #初始化神经网络的线性层权重参数，使用Xavier初始化权重，偏置则初始化为0.01
    def init_weights(self, m):
        if isinstance(m, nn.Linear): #判断是否是线性层
            torch.nn.init.xavier_uniform(m.weight)
            m.bias.data.fill_(0.01)
        
    #输入是两个(点的数量，序列长度，1)，输出(点的数量，序列长度，1)
    def net_u(self, x, t):  
        u = self.dnn(torch.cat([x, t], dim=-1))  #合并为(点的数量，序列长度，2)
        return u
    
    #输入是两个(点的数量，序列长度，1)
    def net_f(self, x, t):
        """ The pytorch autograd version of calculating residual """
        u = self.net_u(x, t) #输出(点的数量，序列长度，1)
        
        #计算u关于t的梯度，也就是u关于t的导数，这里使用了pytorch的自动求导功能
        u_t = torch.autograd.grad(
            u, t,  #输入的张量，要计算u关于t的导数
            grad_outputs=torch.ones_like(u), #生成一个与u形状相同，所有元素均为1的张量，这个参数用于指定向量-雅可比积的像两部分
            retain_graph=True, #表示计算完梯度之后保留计算图若需要多次计算梯度，则需要设置改参数为True
            create_graph=True #创建梯度的计算图，使我们能够计算高阶导数
        )[0] #这个函数的返回值是一个元组，其中包含了每个输入张量的梯度。这里只关心第一个输入张量u的梯度，所以我们使用[0]来获取这个梯度。？？？？又说只有一个梯度

        u_x = torch.autograd.grad(
            u, x,  #输入的张量，要计算u关于x的导数
            grad_outputs=torch.ones_like(u), #生成一个与u形状相同，所有元素均为1的张量，这个参数用于指定向量-雅可比积的像两部分
            retain_graph=True, #表示计算完梯度之后保留计算图若需要多
            create_graph=True #创建梯度的计算图，使我们能够计算高阶导数
        )[0]

        u_xx = torch.autograd.grad(
            u_x, x,  #输入的张量，要计算u_x关于x的
            grad_outputs=torch.ones_like(u_x), #生成一个与u_x形状相同，所有元素均为1的张量，这个参数用于指定向量-雅可比积的像两部分
            retain_graph=True, #表示计算完梯度之后保留计算图若需要多
            create_graph=True #创建梯度的计算图，使我们能够计算高阶导数
        )[0]

        u_tt = torch.autograd.grad(
            u_t, t,  #输入的张量，要计算u_t关于t的
            grad_outputs=torch.ones_like(u_t), #生成一个与u_t形状相同，所有元素均为1的张量，这个参数用于指定向量-雅可比积的像两部分
            retain_graph=True, #表示计算完梯度之后保留计算图若需要多
            create_graph=True #创建梯度的计算图，使我们能够计算高阶导数
        )[0]

        
        f=u_tt - 4 * u_xx #计算f，定义见论文
        return f #(点的数量，序列长度，1)
    
    def loss_func_pretrain(self):
        self.optimizer_LBFGS_pretrain.zero_grad() 
        
        u_pred = self.net_u(self.x_f_pretrain, self.t_f_pretrain) 
        loss = torch.mean((u_pred)**2)
        
        loss.backward() 

        return loss

    def pretrain(self, nIter_pretrain, nIterLBFGS_pretrain):
        print('Start warmup pretraining:')
        print('Start Pretrain Loss: %.4f' % (torch.mean((self.net_u(self.x_f_pretrain, self.t_f_pretrain))**2)))
        for epoch in tqdm(range(nIter_pretrain)):
            self.dnn.train()
            u_pred = self.net_u(self.x_f_pretrain, self.t_f_pretrain) 
            loss = torch.mean((u_pred)**2) 
            
            self.optimizer_Adam_pretrain.zero_grad() 
            loss.backward() 
            self.optimizer_Adam_pretrain.step()  


        for i in tqdm(range(nIterLBFGS_pretrain)):
            self.dnn.train()
            self.optimizer_LBFGS_pretrain.step(self.loss_func_pretrain) 

            
        print('Warmup pretraining end.')
        print('End Pretrain Loss: %.4f' % (torch.mean((self.net_u(self.x_f_pretrain, self.t_f_pretrain))**2)))

    def loss_func(self):
        self.optimizer_LBFGS.zero_grad() 
        
        # u_pred = self.net_u(self.x_u, self.t_u) #调用之前定义的函数，传入参数得到神经网络的输出u
        # if self.config.model_type == 'Transformer':
        #     u_pred = u_pred[:, 0:1].reshape(-1, 1)
        pred_upper = self.net_u(self.x_upper, self.t_upper) #输出(点的数量，序列长度，1)\n",
        pred_lower = self.net_u(self.x_lower, self.t_lower) #输出(点的数量，序列长度，1)\n",
        pred_left = self.net_u(self.x_left, self.t_left) #输出(点的数量，序列长度，1)\n",
        f_pred = self.net_f(self.x_f, self.t_f) #输出(点的数量，序列长度，1)\n",

        pi = torch.tensor(np.pi, dtype=torch.float32, requires_grad=False).to(device)

        #计算损失，共三项损失
        loss_res = torch.mean(f_pred ** 2)
        loss_bc = torch.mean((pred_upper) ** 2) + torch.mean((pred_lower) ** 2)
        ui_t = torch.autograd.grad(pred_left, self.t_left, grad_outputs=torch.ones_like(pred_left), retain_graph=True, create_graph=True)[0]

        loss_ic_1 = torch.mean((pred_left[:,0] - torch.sin(pi*self.x_left[:,0]) - 0.5 * torch.sin(3*pi*self.x_left[:,0])) ** 2)
        loss_ic_2 = torch.mean((ui_t)**2)

        loss_ic = loss_ic_1 + loss_ic_2

        loss = loss_res + loss_bc + loss_ic

        loss.backward()

        return loss #返回loss

      
    
    def train(self, nIter, nIterLBFGS):
        self.dnn.train()#将神经网络设置为训练模式而不是评估模式
        #先使用Adam优化器优化nIter次\n",
        for epoch in tqdm(range(nIter)):
            # u_pred = self.net_u(self.x_u, self.t_u) #调用之前定义的函数，传入参数得到神经网络的输出u
            # if self.config.model_type == 'Transformer':
            #     u_pred = u_pred[:, 0:1].reshape(-1, 1)
            pred_upper = self.net_u(self.x_upper, self.t_upper) #输出(点的数量，序列长度，1)\n",
            pred_lower = self.net_u(self.x_lower, self.t_lower) #输出(点的数量，序列长度，1)\n",
            pred_left = self.net_u(self.x_left, self.t_left) #输出(点的数量，序列长度，1)\n",
            f_pred = self.net_f(self.x_f, self.t_f) #输出(点的数量，序列长度，1)\n",

            pi = torch.tensor(np.pi, dtype=torch.float32, requires_grad=False).to(device)

            #计算损失，共三项损失
            loss_res = torch.mean(f_pred ** 2)
            loss_bc = torch.mean((pred_upper) ** 2) + torch.mean((pred_lower) ** 2)
            ui_t = torch.autograd.grad(pred_left, self.t_left, grad_outputs=torch.ones_like(pred_left), retain_graph=True, create_graph=True)[0]

            loss_ic_1 = torch.mean((pred_left[:,0] - torch.sin(pi*self.x_left[:,0]) - 0.5 * torch.sin(3*pi*self.x_left[:,0])) ** 2)
            loss_ic_2 = torch.mean((ui_t)**2)

            loss_ic = loss_ic_1 + loss_ic_2

            loss = loss_res + loss_bc + loss_ic

            # Backward and optimize
            self.optimizer_Adam.zero_grad() #清除该优化器之前计算的梯度（在PyTorch中，梯度会累积，所以在每次新的优化迭代之前，我们需要清除之前的梯度）\n",
            loss.backward() #被调用以计算损失函数关于神经网络参数的梯度。这个梯度将被用于优化器来更新神经网络参数\n",
            self.optimizer_Adam.step()  #使用之前的优化器self.optimizer_Adam，调用step方法(执行一步优化算法)，传入损失函数self.loss_func，进行优化\n",

            #record the loss value\n",
            self.loss_value.append(loss.item()) #将计算得到的loss值添加到self.loss_value列表中\n",

        #Backward the optimize，使用LBFGS优化器进一步，注意这里虽然迭代了500次，但其实使用LBFGS优化器优化的次数不止500次\n",
        for i in tqdm(range(nIterLBFGS)):
            self.dnn.train() 
            lbfgs_loss = self.optimizer_LBFGS.step(self.loss_func)  
            self.loss_value.append(lbfgs_loss) 
            self.iter += 1

 

  


    def predict(self, X): #形状(点的数量，2)
        if self.config.model_type == 'Transformer':
            X = make_time_sequence(X, num_step=5, step=1e-4) #将形状从(点的数量，2)变为(点的数量，序列长度，2) 
            x = torch.tensor(X[:,:, 0:1], dtype=torch.float32, requires_grad=True).to(device) #(点的数量，序列长度，1) 
            t = torch.tensor(X[:,:, 1:2], dtype=torch.float32, requires_grad=True).to(device)
        
        if self.config.model_type == 'DNN' or self.config.model_type == 'KAN':
            x = torch.tensor(X[:,0:1], dtype=torch.float32, requires_grad=True).to(device) #(点的数量，序列长度，1) 
            t = torch.tensor(X[:,1:2], dtype=torch.float32, requires_grad=True).to(device)

        self.dnn.eval() #将神经网络切换为评估模式
        with torch.no_grad():
            u = self.net_u(x, t) #形状从两个(点的数量，序列长度，1)变为(点的数量，序列长度，1)
            if self.config.model_type == 'Transformer':
                u = u[:,0:1].reshape(-1,1) #将u的形状从(点的数量，序列长度，1)变为(点的数量，1)
            u = u.detach().cpu().numpy() #将张量u和f先从计算图中分离出来，然后转换为numpy数组，最后将这个数组移动到cpu上

        return u #u形状是(点的数量，1)，f形状是(点的数量，1)
    
    
    def residual(self, X): #形状(点的数量，2)
        if self.config.model_type == 'Transformer':
            X = make_time_sequence(X, num_step=5, step=1e-4) #将形状从(点的数量，2)变为(点的数量，序列长度，2) 
            x = torch.tensor(X[:,:, 0:1], dtype=torch.float32, requires_grad=True).to(device) #(点的数量，序列长度，1) 
            t = torch.tensor(X[:,:, 1:2], dtype=torch.float32, requires_grad=True).to(device)
        if self.config.model_type == 'DNN' or self.config.model_type == 'KAN':
            x = torch.tensor(X[:,0:1], dtype=torch.float32, requires_grad=True).to(device) #(点的数量，序列长度，1) 
            t = torch.tensor(X[:,1:2], dtype=torch.float32, requires_grad=True).to(device)

        self.dnn.eval() #将神经网络切换为评估模式
        
        f = self.net_f(x, t) #形状从两个(点的数量，序列长度，1)变为(点的数量，序列长度，1)
        if self.config.model_type == 'Transformer':
            f = torch.mean(f, dim=1).reshape(-1,1) #对f在序列长度维度上求平均，得到(点的数量，1)，即每个点的平均残差
        # f = f[:,0:1].reshape(-1,1) #将f的形状从(点的数量，序列长度，1)变为(点的数量，1)
        f = f.detach().cpu().numpy()
        return f #u形状是(点的数量，1)，f形状是(点的数量，1)
    

    def hidden_predict_transformer(self, X):#形状(点的数量，序列长度，2)
        x = torch.tensor(X[:,:, 0:1], requires_grad=True).float().to(device) #(点的数量，序列长度，1) 
        t = torch.tensor(X[:,:, 1:2], requires_grad=True).float().to(device)
        self.dnn.eval()
        with torch.no_grad():
            hidden_output = self.dnn.hidden_output(torch.cat([x, t], dim=-1)) #(点的数量，序列长度，2)
            hidden_output = hidden_output.detach().cpu().numpy()
        # hidden_output_x = hidden_output[:, 0]
        # hidden_output_t = hidden_output[:, 1]
        # hidden_output_x = hidden_output_x.detach().cpu().numpy()
        # hidden_output_t = hidden_output_t.detach().cpu().numpy()
        return hidden_output #(点的数量，序列长度，2)
    
    def hidden_predict_dnn(self, x, t):
        x = torch.tensor(x, requires_grad=True).float().to(device) #从输入中得到x和t（第一列和第二列），是张量，需要计算梯度，转换为浮点数类型，并将张量移动到指定设备上
        t = torch.tensor(t, requires_grad=True).float().to(device)
        self.dnn.eval()
        hidden_output = self.dnn.hidden_output(torch.cat([x, t], dim=1))
        hidden_output_x = hidden_output[:, 0]
        hidden_output_t = hidden_output[:, 1]
        hidden_output_x = hidden_output_x.detach().cpu().numpy()
        hidden_output_t = hidden_output_t.detach().cpu().numpy()
        return hidden_output_x, hidden_output_t #(点的数量，) #(点的数量，)



    

In [23]:
#定义设置随机数种子的函数，第一个参数seed表示种子；第二个参数用来设置CUDA的卷积操作是否确定性，默认为False，表示没有确定性
def set_seed(seed):
    # torch.manual_seed(seed) #设置pytorch的CPU随机数生成器的种子
    # torch.cuda.manual_seed_all(seed) #设置putorch的所有GPU随机数生成器的种子
    # np.random.seed(seed) #设置numpy的随机数生成器的种子
    # random.seed(seed) #设置python的内置随机数生成器的种子
    # torch.backends.cudnn.deterministic = deterministic #True会让CUDA的卷积操作变得确定性，即对于相同的输入，每次运行会得到相同的结果，False则相反
    """
    设置PyTorch的随机种子, 用于生成随机数. 通过设置相同的种子, 可以确保每次运行时生成的随机数序列相同
    """
    torch.manual_seed(seed)
 
    """
    设置PyTorch在所有可用的CUDA设备上的随机种子. 如果在使用GPU进行计算, 这个设置可以确保在不同的GPU上生成的随机数序列相同
    """
    torch.cuda.manual_seed_all(seed)
 
    """
    设置PyTorch在当前CUDA设备上的随机种子. 它与上一行代码的作用类似, 但只影响当前设备
    """
    torch.cuda.manual_seed(seed)
 
    """
    设置NumPy的随机种子, 用于生成随机数. 通过设置相同的种子，可以确保在使用NumPy的随机函数时生成的随机数序列相同
    """
    np.random.seed(seed)
    
    """
    设置Python内置的随机函数的种子. Python的random模块提供了许多随机函数, 包括生成随机数、打乱列表等. 通过设置相同的种子, 可以确保使用这些随机函数时生成的随机数序列相同
    """
    random.seed(seed)
    
    """
    设置Python的哈希种子 (哈希函数被广泛用于数据结构 (如字典和集合) 的实现，以及一些内部操作 (如查找和比较)). 通过设置相同的种子, 可以确保在不同的运行中生成的哈希结果相同
    """
    # os.environ["PYTHONHASHSEED"] = str(seed)
    
    """
    该设置确保每次运行代码时, cuDNN的计算结果是确定性的, 即相同的输入会产生相同的输出, 这是通过禁用一些非确定性的算法来实现的, 例如在卷积操作中使用的算法. 这样做可以保证模型的训练和推理在相同的硬件和软件环境下是可复现的, 即每次运行代码时的结果都相同. 但是, 这可能会导致一些性能上的损失, 因为禁用了一些优化的非确定性算法
    """
    torch.backends.cudnn.deterministic = True
    
    """
    该设置禁用了cuDNN的自动优化过程. 当它被设置为False时, PyTorch不会在每次运行时重新寻找最优的算法配置, 而是使用固定的算法配置. 这样做可以确保每次运行代码时的性能是一致的, 但可能会导致一些性能上的损失
    """
    torch.backends.cudnn.benchmark = False



In [24]:
#定义根据模型计算给定输入（点集中的点）的混沌度的函数，这个函数接受三个参数，分别是模型、输入数据和迭代次数
def calculate_chaos_transformer(model, X, num_iter): #输入是(点的数量，2)
    """
    计算模型混沌情况的函数。
    参数:
    - model: 用于预测的模型对象，必须有一个名为hidden_predict的方法。模型的hidden_predict为倒数第二层的输出，倒数第二层的维度必须和输入维度相同。
    - X: 输入数据，形状为(N_f_new, 2)，其中第一列为x0，第二列为t0。
    - num_iter: 计算混沌情况的迭代次数。
    返回:
    - distances: 每个采样点（与该采样点加上微扰比较）在最后一次迭代后的欧氏距离数组，形状为(N_f_new,)。
    """
    #对于所有的采样点
    X0 = X
    x0 = X[:, 0:1] #取X_f_train_new的第一列，赋值给x0，(N_f_new,1)形状
    t0 = X[:, 1:2] #取X_f_train_new的第二列，赋值给t0
    xs = []
    X0 = make_time_sequence(X0, num_step=5, step=1e-4) #将形状从(点的数量，2)变为(点的数量，序列长度，2)
    X = model.hidden_predict_transformer(X0) #(点的数量，序列长度，2)
    
    for i in range(num_iter): #循环num_iter次
        X = model.hidden_predict_transformer(X) #(点的数量，序列长度，2)
        xs.append(X) #新添加(点的数量，序列长度，2)

    # 给所有采样点加上一个很小的扰动
    x1 = x0 + np.random.normal(0, 0.0001) #加上一个很小的扰动，(N_f_new,1)形状
    t1 = t0 + np.random.normal(0, 0.0001)
    # 利用x0{1}和t0{1}计算x{t1}和t{t1}，存储在xs1中
    xs1 = [] #初始化xs1
    X1 = np.concatenate((x1, t1), axis=1) #(N_f_new,2)形状
    X1 = make_time_sequence(X1, num_step=5, step=1e-4) #将形状从(点的数量，2)变为(点的数量，序列长度，2)
    X1 = model.hidden_predict_transformer(X1) #(N_f_new,序列长度,2)

    for i in range(num_iter): #循环num_iter次
        X1 = model.hidden_predict_transformer(X1) #(N_f_new,序列长度,
        xs1.append(X1) #新添加(N_f_new,序列长度,2)

    

    # # 计算最后一次迭代的隐藏层输出，即最后一次迭代的x和t,使用chunchaos要注释掉
    # last_iter_xs = np.array(xs[-1]) #转换为数组，便于之后计算距离
    # last_iter_xs1 = np.array(xs1[-1])
    # distances = np.linalg.norm(last_iter_xs - last_iter_xs1, axis=(1,2))
    # distances = distances.flatten()







    #chunchaos方法
    num_elements_to_average = 20
    # 提取最后 20 个元素
    last_20_xs = xs[-num_elements_to_average:] # 这是一个包含 20 个元素的列表
    last_20_xs1 = xs1[-num_elements_to_average:] # 这是一个包含 20 个元素的列表
    # 存储每个时间步（列表中的每个元素）的混沌度分数
    step_chaos_scores = []
    # 遍历最后 20 个元素,zip 将两个列表的对应元素配对
    for step_xs, step_xs1 in zip(last_20_xs, last_20_xs1):
        # step_xs 和 step_xs1 都是形状为 (N_f_new, sequence_length, 2) 的数组/张量
        difference_step = step_xs - step_xs1 # 计算差异
        # 计算每个数据点在当前步的混沌度分数 (形状 (N_f_new,))
        # 假设使用 NumPy 进行计算
        step_scores = np.linalg.norm(difference_step, axis=(1, 2))
        # 将这 N_f_new 个分数添加到列表中
        step_chaos_scores.append(step_scores)

    # step_chaos_scores 是一个包含 20 个元素的列表，每个元素都是形状为 (N_f_new,) 的 NumPy 数组
    # 将列表转换成一个单一的 NumPy 数组，形状为 (20, N_f_new)
    all_step_scores = np.stack(step_chaos_scores)
    # 现在 all_step_scores 的形状是 (num_elements_to_average, N_f_new)
    # axis 0 是时间步 (20)，axis 1 是原始数据点 (N_f_new)
    # 计算最后 20 个时间步在每个数据点上的平均混沌度分数
    # 沿着时间步轴 (axis=0) 求平均
    distances = np.mean(all_step_scores, axis=0)
    # 结果形状是 (N_f_new,)






    return distances

In [25]:
import numpy as np

def calculate_chaos_dnn(model, X, num_iter, last_k=20):
    """
    计算模型在给定输入点集上的混沌度。
    参数:
    - model: 具有 hidden_predict 方法的神经网络模型，返回倒数第二层输出
    - X: 输入数据，形状为 (N_f_new, 2)，其中第一列为 x0，第二列为 t0
    - num_iter: 计算混沌情况的总迭代次数
    - last_k: 计算混沌度时考虑的最后 k 次迭代

    返回:
    - chaos_score: 每个采样点的混沌度（取最后 k 次的平均欧氏距离）
    """

    # 1. 取输入点 (x0, t0)
    x0 = X[:, 0:1]  # (N_f_new,1)
    t0 = X[:, 1:2]  # (N_f_new,1)

    # 2. 计算初始点的迭代轨迹
    xs = np.zeros((num_iter, X.shape[0], 2))  # 形状 (num_iter, N_f_new, 2)
    x, t = model.hidden_predict_dnn(x0, t0)
    x, t = x.reshape(-1,1), t.reshape(-1,1)

    for i in range(num_iter):
        x, t = model.hidden_predict_dnn(x, t)
        x, t = x.reshape(-1,1), t.reshape(-1,1)
        xs[i, :, 0] = x.flatten()  # 存储 x
        xs[i, :, 1] = t.flatten()  # 存储 t

    # 3. 给输入数据加微小扰动
    x1 = x0 + np.random.normal(0, 0.0001, size=x0.shape)
    t1 = t0 + np.random.normal(0, 0.0001, size=t0.shape)

    # 4. 计算扰动点的迭代轨迹
    xs1 = np.zeros_like(xs)  # 形状 (num_iter, N_f_new, 2)
    x, t = model.hidden_predict_dnn(x1, t1)
    x, t = x.reshape(-1,1), t.reshape(-1,1)

    for i in range(num_iter):
        x, t = model.hidden_predict_dnn(x, t)
        x, t = x.reshape(-1,1), t.reshape(-1,1)
        xs1[i, :, 0] = x.flatten()
        xs1[i, :, 1] = t.flatten()

    # 5. 计算最后 last_k 次迭代的欧氏距离
    last_xs = xs[-last_k:, :, :]  # 取最后 k 次迭代的结果
    last_xs1 = xs1[-last_k:, :, :]  # 取扰动点的结果

    # 计算最后 k 次的欧氏距离
    distances = np.linalg.norm(last_xs - last_xs1, axis=2)  # 形状 (last_k, N_f_new)

    # 6. 取最后 k 次迭代的平均混沌度
    chaos_score = np.mean(distances, axis=0)  # (N_f_new,)

    return chaos_score


In [26]:
config.train_method

'Standard'

In [27]:
# #RAR-G方法，对1000个点，先选择10个点训练500次，然后每500次迭代重采样100个点，选出其中残差最大的10个点添加到训练点中；最后总共有1000个点，共训练10000次
seeds = [0, 1, 12, 33, 123, 321, 1234, 4321, 12345, 54321] #生成10个随机种子\n",
# seeds = [0,1]

lb = np.array([0.0, 0.0])
ub = np.array([2 * np.pi, 1.0])

# Train PINNsformer
_, b_left, b_right, b_upper, b_lower = get_data([0,2*np.pi], [0,1], 51, 51) #51代表初值点、边界值点数量！！
X_star, _, _, _, _ = get_data([0,2*np.pi], [0,1], 101, 101) #生成测试数据，采样点数为101x101,这里的X_star就是res_test

error_u = [] #创建一个空列表，用于存储误差值
error_mae = [] #创建一个空列表，用于存储MAE值
error_mse = [] #创建一个空列表，用于存储MSE值

i = 0 #初始化i为0
def u_ana(x,t):
    return np.sin(np.pi*x) * np.cos(2*np.pi*t) + 0.5 * np.sin(3*np.pi*x) * np.cos(6*np.pi*t)





import pandas as pd
all_records = []





for seed in seeds:
    set_seed(seed) #设置随机种子
    
    # 在 seeds 循环外部定义（或者每个种子独立，在循环内初始化）
    test_pred_history = []      # 存储 (iter_count, prediction_array) 元组
    global_iter = 0             # 全局迭代计数器（用于记录训练步数）
    

    # if config.train_method == 'Standard':

    #     X_f_train = lb + (ub-lb)*lhs(2, config.N_f) #lhs函数采用拉丁超采样方法，生成一个近似均匀分布的多维样本点集，返回的是一个形状为（$N_f$，2）的数组，每一行都是一个2维的样本点，所有样本点都在[0,1]范围内，并对该样本集进行缩放，把每个样本从[0,1]区间缩放到[lb,ub]区域内，即得到了指定范围内均匀分布的样本$X_f$。
    #     X_f_pretrain = lb + (ub-lb)*lhs(2, config.N_f_pretrain) #生成预训练数据，采样点数为N_f_pretrain

    #     #创建PINN模型并输入各种参数      
    #     model = PhysicsInformedNN(config, b_left, b_right, b_upper, b_lower, X_f_train, X_f_pretrain)
    #     #获取当前时间并赋值给start_time   
    #     start_time = time.time()      
    #     #训练模型50000次    
    #     if config.use_pretrained:
    #         model.pretrain(config.nIter_pretrain, config.nIterLBFGS_pretrain) #预训练   
    #     model.train(config.nIter, config.nIterLBFGS)
    #     #获取当前时间并减去start_time，得到训练时间并赋值给elapsed
    #     elapsed = time.time() - start_time                
    #     #打印训练所需时间\n",
    #     print('Training time: %.4f' % (elapsed))

    if config.train_method == 'Standard':
            X_f_train = lb + (ub-lb)*lhs(2, config.N_f)
            X_f_pretrain = lb + (ub-lb)*lhs(2, config.N_f_pretrain)
            model = PhysicsInformedNN(config, b_left, b_right, b_upper, b_lower, X_f_train, X_f_pretrain)
            start_time = time.time()
            
            if config.use_pretrained:
                model.pretrain(config.nIter_pretrain, config.nIterLBFGS_pretrain)
            
            num_record_segments = 20
            
            if config.opt_list == 'Adam+LBFGS':
                # Adam 拆段
                if config.nIter > 0:
                    seg_iters = config.nIter // num_record_segments
                    for _ in range(num_record_segments):
                        model.train(seg_iters, 0)
                        global_iter += seg_iters
                        test_pred_history.append((global_iter, model.predict(X_star)))
                    remainder = config.nIter - seg_iters * num_record_segments
                    if remainder > 0:
                        model.train(remainder, 0)
                        global_iter += remainder
                        test_pred_history.append((global_iter, model.predict(X_star)))
                # LBFGS 一次跑完
                if config.nIterLBFGS > 0:
                    model.train(0, config.nIterLBFGS)
                    global_iter += config.nIterLBFGS
                    test_pred_history.append((global_iter, model.predict(X_star)))
            
            elif config.opt_list == 'LBFGS':
                # 拆 LBFGS（前提：train 内部不重建 optimizer_LBFGS）
                seg_iters = config.nIterLBFGS // num_record_segments
                for _ in range(num_record_segments):
                    model.train(0, seg_iters)
                    global_iter += seg_iters
                    test_pred_history.append((global_iter, model.predict(X_star)))
                remainder = config.nIterLBFGS - seg_iters * num_record_segments
                if remainder > 0:
                    model.train(0, remainder)
                    global_iter += remainder
                    test_pred_history.append((global_iter, model.predict(X_star)))
            
            elapsed = time.time() - start_time


    else:
        # --- 1. 参数设置 ---
        # if config.opt_list == 'LBFGS':
        #     config.nIter = 0
        #     config.nIterLBFGS = 600

        # --- 2. 初始化 ---
        N_f_1 = config.N_f // 20
        X_f_train = lb + (ub - lb) * lhs(2, N_f_1)
        X_f_pretrain = lb + (ub - lb) * lhs(2, config.N_f_pretrain) #生成预训练数据，采样点数为N_f_pretrain
        model = PhysicsInformedNN(config, b_left, b_right, b_upper, b_lower, X_f_train, X_f_pretrain)

        # 【新】创建巨大、固定的候选点池
        N_f_candidate = config.N_f * 2
        X_f_candidate = lb + (ub - lb) * lhs(2, N_f_candidate)
        is_point_selected_mask = np.zeros(N_f_candidate, dtype=bool) #布尔掩码，形状是(N_f_candidate,)，初始值为False，表示所有点都未被选择

        start_time = time.time()      
        if config.use_pretrained:
            model.pretrain(config.nIter_pretrain, config.nIterLBFGS_pretrain) #预训练

        #训练模型50000次       
        if config.opt_list == 'Adam+LBFGS':
            model.train(config.nIter // 20, 0)
            # --- 4. 主动学习循环 ---
            # 计算总轮数,这里是19
            num_active_rounds = (config.nIter - config.nIter // 20) // (config.nIter // 20)
            # 每轮训练中记录的检查点数量，其实就是后面的T
            num_checkpoints_per_round = 10
            # 两次检查点之间的训练迭代次数
            epochs_between_checkpoints = (config.nIter // 20) // num_checkpoints_per_round
        elif config.opt_list == 'LBFGS':
            model.train(0, config.nIterLBFGS//20)
            if config.model_type == 'KAN':
                model.optimizer_LBFGS = torch.optim.LBFGS(model.dnn.parameters(), line_search_fn='strong_wolfe', tolerance_grad = 1e-8, tolerance_change= 1e-10)
            else:
                model.optimizer_LBFGS = torch.optim.LBFGS(model.dnn.parameters(), line_search_fn='strong_wolfe') 
            # --- 4. 主动学习循环 ---
            # 计算总轮数,这里是19
            num_active_rounds = (config.nIterLBFGS - config.nIterLBFGS // 20) // (config.nIterLBFGS // 20)
            # 每轮训练中记录的检查点数量，其实就是后面的T
            num_checkpoints_per_round = 10
            # 两次检查点之间的训练迭代次数
            epochs_between_checkpoints = (config.nIterLBFGS // 20) // num_checkpoints_per_round
                

        # 用于存储每一轮最终预测的全局历史
        global_final_predictions = []

        for i in range(num_active_rounds):        
            # a. 训练并记录预测历史
            prediction_history = []
            # 记录当前模型在整个候选池上的预测输出
            
            for j in range(num_checkpoints_per_round):
                if config.opt_list == 'Adam+LBFGS':
                    model.train(epochs_between_checkpoints, 0)



                    # 记录测试集预测
                    global_iter += epochs_between_checkpoints   # 根据实际训练迭代数累加
                    pred_test = model.predict(X_star)
                    test_pred_history.append((global_iter, pred_test))
                elif config.opt_list == 'LBFGS':
                    model.train(0, epochs_between_checkpoints)



                    # 记录测试集预测
                    global_iter += epochs_between_checkpoints   # 根据实际训练迭代数累加
                    pred_test = model.predict(X_star)
                    test_pred_history.append((global_iter, pred_test))

                # 记录当前模型在整个候选池上的预测输出
                predictions = model.predict(X_f_candidate) # 返回 (N, 1) 的Numpy数组
                # prediction_history.append(predictions.flatten()) # 存为一维数组，每个元素形状为 (N, ) 
                prediction_history.append(predictions) # 存为二维数组，每个元素形状为 (N, 1)
            local_prediction_history = prediction_history
            if config.opt_list == 'LBFGS':
                if config.model_type == 'KAN':
                    model.optimizer_LBFGS = torch.optim.LBFGS(model.dnn.parameters(), line_search_fn='strong_wolfe', tolerance_grad = 1e-8, tolerance_change= 1e-10)
                else:
                    model.optimizer_LBFGS = torch.optim.LBFGS(model.dnn.parameters(), line_search_fn='strong_wolfe') 
        

            # b. 计算不稳定性分数 g(x) = Σ v_t * a_t(x)
            # 将历史记录转为矩阵 (T+1, N_candidates,1)
            local_history_matrix = np.array(local_prediction_history)
            T_local, num_candidates, _ = local_history_matrix.shape #T表示共存储了多少个模型的预测结果，num_candidates表示候选点的数量

            # 获取最终预测u_T(x)
            local_final_prediction = local_history_matrix[-1, :, :] # (N_candidates, 1) 的数组

            # 计算不一致性矩阵 a_t(x) = |u_t(x) - u_T(x)|
            # 利用 NumPy 的广播机制高效计算，具体为(T, N_candidates,1)-(N_candidates,1)（相当于用T个元素分别和最后一个元素相减）,最后得到的形状是(T, N_candidates)
            # disagreement_matrix_A = np.abs(prediction_history_matrix - final_prediction_vec)
            local_disagreement = np.linalg.norm(local_history_matrix - local_final_prediction, ord=2, axis=-1)

            # 计算时间权重向量 v_t = (t/T)^k
            t_local = np.arange(1, T_local + 1) #形状是(T,)的数组
            k_local = 1.0  # 权重指数，可以作为超参数调整
            v_local = (t_local / T_local) ** k_local #形状是(T,)

            # 计算最终不稳定性分数 g(x)
            # g = V^T * A (矩阵乘法概念)
            # 实际操作：将权重向量V变形以进行逐元素乘法，然后在时间轴上求和
            # v_local（也就是V）的形状是(T,)，local_disagreement（也就是A）的形状是(T, N_candidates)
            # V[:, np.newaxis] 将V变为 (T, 1)，可以与A进行广播乘法
            g_micro = np.sum(v_local[:, np.newaxis] * local_disagreement, axis=0) #求和前形状为 (T, N_candidates)，求和后形状为(N_candidates,)

            
            # c. 更新并计算【宏观不稳定性 g_macro】
            # ------------------------------------------------------------------
            # 将本轮的最终预测加入全局历史列表
            global_final_predictions.append(local_final_prediction)
            
            g_macro = np.zeros(num_candidates) # 如果历史太短，宏观不稳定性为0
            
            # 只有当全局历史足够长时（至少有两次最终预测），计算才有意义
            if len(global_final_predictions) > 1:
                global_history_matrix = np.array(global_final_predictions)
                T_global, _, _ = global_history_matrix.shape # T_global 是当前的主动学习轮数 i+1
                
                # 使用全局最新的预测作为基准
                global_final_prediction = global_history_matrix[-1, :, :]
                
                # 计算宏观不一致性
                global_disagreement = np.linalg.norm(global_history_matrix - global_final_prediction, ord=2, axis=-1)
                
                # 【宏观 t, k】
                t_global = np.arange(1, T_global + 1)
                k_macro = 2.0 # 宏观时间尺度上，可以给后期变化更大的惩罚，所以k可以更大
                v_global = (t_global / T_global) ** k_macro
                
                # 计算宏观不稳定性分数
                g_macro = np.sum(v_global[:, np.newaxis] * global_disagreement, axis=0)


            # d. 融合分数
            # ------------------------------------------------------------------
            # 在融合前，对两种分数进行归一化，使其尺度相当，避免其中一个主导
            # 这是一个很好的实践，可以防止数值问题
            if np.std(g_micro) > 1e-6:
                g_micro_norm = (g_micro - np.mean(g_micro)) / np.std(g_micro)
            else:
                g_micro_norm = np.zeros_like(g_micro)

            if np.std(g_macro) > 1e-6:
                g_macro_norm = (g_macro - np.mean(g_macro)) / np.std(g_macro)
            else:
                g_macro_norm = np.zeros_like(g_macro)
                
            # 设置融合权重
            w_micro = 0.5
            w_macro = 0.5
            
            instability_scores_g = w_micro * g_micro_norm + w_macro * g_macro_norm

            epsilon = 1e-5

            if config.model_type == 'Transformer':
                num_iter=100
                #计算混沌度
                distances = calculate_chaos_transformer(model, X_f_candidate, num_iter)
                norm_distances = np.linalg.norm(distances)
                if norm_distances > epsilon:
                    distances = distances / norm_distances
                else:
                    distances = np.zeros_like(distances)
            elif config.model_type == 'DNN':
                num_iter=100
                #计算混沌度
                distances = calculate_chaos_dnn(model, X_f_candidate, num_iter)
                norm_distances = np.linalg.norm(distances)
                if norm_distances > epsilon:
                    distances = distances / norm_distances
                else:
                    distances = np.zeros_like(distances)


            if config.model_type == 'Transformer':
                batch_size = N_f_candidate // 10 
                all_residuals = [] 
                for i in range(0, N_f_candidate, batch_size):
                    X_batch = X_f_candidate[i : i + batch_size]
                    residual_batch = model.residual(X_batch) 
                    all_residuals.append(residual_batch)
                residual = np.vstack(all_residuals)
                abs_residual = np.abs(residual)
                abs_residual = abs_residual.flatten()
            else:
                residual = model.residual(X_f_candidate)
                abs_residual = np.abs(residual).flatten()


            #进行归一化
            
            
            norm_residual = np.linalg.norm(abs_residual)
            if norm_residual > epsilon:
                abs_residual = abs_residual / norm_residual
            else:
                abs_residual = np.zeros_like(abs_residual)

            #进行归一化
            norm_instability = np.linalg.norm(instability_scores_g)
            if norm_instability > epsilon:
                instability_scores_g = instability_scores_g / norm_instability
            else:
                instability_scores_g = np.zeros_like(instability_scores_g)

            if config.train_method == 'Residual':
                xinxi = abs_residual
            elif config.train_method == 'Chaos':
                xinxi = distances
            elif config.train_method == 'Failure':
                xinxi = instability_scores_g
            elif config.train_method == 'Failure + Residual + Chaos':
                xinxi = abs_residual + instability_scores_g + distances
            elif config.train_method == 'Failure + Residual':
                xinxi = abs_residual + instability_scores_g
            # xinxi = distances + abs_residual + instability_scores_g # 计算综合分数，距离、残差和不稳定性分数越大，综合分数越大

            # e. 选择新的点
            # 屏蔽已选中的点，防止重复选择
            xinxi[is_point_selected_mask] = -1.0
            
            # 找出分数最高的 N_f_1 个点的索引
            topk_indices = np.argpartition(xinxi, -N_f_1)[-N_f_1:]
            
            # 提取新点
            X_f_train_topk = X_f_candidate[topk_indices]
            
            # f. 更新训练集和掩码
            X_f_train = np.vstack((X_f_train, X_f_train_topk))
            is_point_selected_mask[topk_indices] = True


            # e. 更新模型内部的配位点
            if config.model_type == 'Transformer':
                X_f = make_time_sequence(X_f_train, num_step=5, step=1e-4)
                #配位点数据
                model.x_f = torch.tensor(X_f[:,:, 0:1], requires_grad=True).float().to(device)
                model.t_f = torch.tensor(X_f[:,:, 1:2], requires_grad=True).float().to(device)
            else:
                X_f = X_f_train
                #配位点数据
                model.x_f = torch.tensor(X_f[:, 0:1], requires_grad=True).float().to(device)
                model.t_f = torch.tensor(X_f[:, 1:2], requires_grad=True).float().to(device)


        if config.opt_list == 'LBFGS':
            model.train(0, config.nIterLBFGS-200)




            # 第一段训练
            global_iter += (config.nIterLBFGS - 200)
            test_pred_history.append((global_iter, model.predict(X_star)))

            
        elif config.opt_list == 'Adam+LBFGS': 
            model.train(0, config.nIterLBFGS)


            # 第二段训练
            global_iter += config.nIterLBFGS
            test_pred_history.append((global_iter, model.predict(X_star)))  
        
        #获取当前时间并减去start_time，得到训练时间并赋值给elapsed
        elapsed = time.time() - start_time                
        #打印训练所需时间\n",
        print('Training time: %.4f' % (elapsed))


    #用训练好的模型进行预测，返回四个值（均为数组）  
    u_pred = model.predict(X_star).flatten()

    # u_pred = u_pred.reshape(101,101)

    u_star = u_ana(X_star[:,0], X_star[:,1])
    # .reshape(101,101)\n",


    #计算误差（基于2范数）        
    error_u.append(np.linalg.norm(u_star-u_pred,2)/np.linalg.norm(u_star,2)) #计算误差，然后将误差添加到error_u列表中             
    # 计算 MAE 和 MSE
    mae = np.mean(np.abs(u_star - u_pred))
    mse = np.mean((u_star - u_pred) ** 2)
    # 记录 MAE 和 MSE
    error_mae.append(mae)
    error_mse.append(mse)

    i+=1 #i加1
    print(f'当前为第{i}次循环，种子为{seed}')
    print('Error u : %e' % (np.linalg.norm(u_star-u_pred,2)/np.linalg.norm(u_star,2))) #打印误差  
    print('MAE: %e' % mae) #打印MAE
    print('MSE: %e' % mse) #打印MSE

    # 提取记录（按迭代步数排序，但记录本身就是按时间顺序，无需再排序）
    recorded_iters = [item[0] for item in test_pred_history]
    recorded_preds = [item[1] for item in test_pred_history]

    # if len(recorded_preds) >= 50:
    #     # 等间隔选择索引
    #     indices = np.linspace(0, len(recorded_preds)-1, 50, dtype=int)
    # else:
    indices = np.arange(len(recorded_preds))  # 不够则全部使用
    selected_preds = [recorded_preds[i] for i in indices]
    selected_iters = [recorded_iters[i] for i in indices]

    history_matrix = np.array(selected_preds)  # (T_selected, N_test, 1)
    T_sel, N_test, _ = history_matrix.shape
    final_pred = history_matrix[-1, :, :]      # 最后一次预测（即最终模型）

    # 不一致性矩阵 |u_t - u_T|
    disagreement = np.linalg.norm(history_matrix - final_pred, ord=2, axis=-1)  # (T_sel, N_test)

    # 时间权重 v_t = (t/T)^k (t从1开始)
    t_seq = np.arange(1, T_sel + 1)
    k = 1.0  # 与原代码保持一致
    v = (t_seq / T_sel) ** k

    # Failure 分数
    failure_test = np.sum(v[:, np.newaxis] * disagreement, axis=0)  # (N_test,)
    print('总failure值 : %e' % (np.mean(failure_test))) #打印误差  
    mean_failure = float(np.mean(failure_test))   # 在升序保存前定义一次即可
    # 残差
    # 残差
    if config.model_type == 'Transformer':
        batch_size = X_star.shape[0] // 10 
        all_residuals = [] 
        for i in range(0, X_star.shape[0], batch_size):
            X_batch = X_star[i : i + batch_size]
            residual_batch = model.residual(X_batch) 
            all_residuals.append(residual_batch)
        residual = np.vstack(all_residuals)
        abs_residual = np.abs(residual)
        residual_test = abs_residual.flatten()
    else:
        residual = model.residual(X_star)
        residual_test = np.abs(residual).flatten()

    # # 混沌度（根据您的模型类型）
    # if config.model_type == 'Transformer':
    #     chaos_test = calculate_chaos_transformer(model, X_star, 100)
    # else:
    #     chaos_test = calculate_chaos_dnn(model, X_star, 100)

    # xinxi_test = residual_test
    xinxi_test = failure_test
    # xinxi_test = chaos_test
    # 全量误差
    u_pred_full = model.predict(X_star).flatten()
    u_star_full = u_ana(X_star[:,0], X_star[:,1])
    # ========== 多指标、多删除比例的 L2 误差与提升对比 ==========
    # 假设已有变量：residual_test, chaos_test, failure_test (一维数组)
    # 以及 u_pred_full, u_star_full, seed

    remove_ratios = np.arange(0, 1.0, 0.05)   # 0%, 5%, ..., 95%
    N_test = len(u_star_full)

    # 定义指标及其评分
    metrics = {
        'Residual': residual_test,
        # 'Chaos': chaos_test,
        'Failure': failure_test   # 若无，可注释掉
    }

    results = {}  # {指标名: {'L2': [], '提升%': []}}



    for name, scores in metrics.items():
        # sorted_idx = np.argsort(scores)[::-1]  # 降序（高风险）
        sorted_idx = np.argsort(scores) #升序
        l2_list = []
        for ratio in remove_ratios:
            num_remove = int(N_test * ratio)
            keep_idx = sorted_idx[num_remove:]
            if len(keep_idx) == 0:
                l2_list.append(np.nan)
            else:
                u_pred_keep = u_pred_full[keep_idx]
                u_star_keep = u_star_full[keep_idx]
                l2 = np.linalg.norm(u_star_keep - u_pred_keep, 2) / np.linalg.norm(u_star_keep, 2)
                l2_list.append(l2)
        # 计算提升百分比（相对于删除比例为0%时的L2误差）
        l2_0 = l2_list[0]  # 删除0%的L2
        if np.isnan(l2_0) or l2_0 == 0:
            improvement = [np.nan] * len(l2_list)
        else:
            improvement = [(l2_0 - l2) / l2_0 * 100 if not np.isnan(l2) else np.nan for l2 in l2_list]
        results[name] = {'L2': l2_list, '提升%': improvement}

    for name, res in results.items():
        for r, l2, imp in zip(remove_ratios, res['L2'], res['提升%']):
            all_records.append({
                'seed': seed, 'metric': name, 'order': 'asc',
                'remove_ratio': r, 'L2': l2, 'improvement_pct': imp,
                'mean_failure': mean_failure,
            })

    # 打印表格
    try:
        import pandas as pd
        df = pd.DataFrame({'删除比例(%)': (remove_ratios * 100).astype(int)})
        for name in metrics.keys():
            df[f'{name}_L2'] = results[name]['L2']
            df[f'{name}_提升%'] = results[name]['提升%']
        print('='*80)
        print(f'【种子 {seed}】各指标升序排列不同删除比例下的 L2 误差及相比不删除的提升百分比')
        print(df.round(6).to_string(index=False))
    except ImportError:
        # 无 pandas 时简单打印
        print('='*80)
        print(f'【种子 {seed}】各指标升序排列不同删除比例下的 L2 误差')
        print('删除比例%  ' + '  '.join(metrics.keys()))
        for i, ratio in enumerate(remove_ratios):
            row = f'{int(ratio*100):>6}%  '
            for name in metrics.keys():
                row += f'{results[name]["L2"][i]:.6f}  '
            print(row)
        # 打印提升
        print('\n提升百分比 (%)')
        print('删除比例%  ' + '  '.join(metrics.keys()))
        for i, ratio in enumerate(remove_ratios):
            row = f'{int(ratio*100):>6}%  '
            for name in metrics.keys():
                row += f'{results[name]["提升%"][i]:.2f}  '
            print(row)
    print('='*80)
    # ========== 多指标、多删除比例的 L2 误差与提升对比 ==========
    # 假设已有变量：residual_test, chaos_test, failure_test (一维数组)
    # 以及 u_pred_full, u_star_full, seed

    remove_ratios = np.arange(0, 1.0, 0.05)   # 0%, 5%, ..., 95%
    N_test = len(u_star_full)

    # 定义指标及其评分
    metrics = {
        'Residual': residual_test,
        # 'Chaos': chaos_test,
        'Failure': failure_test   # 若无，可注释掉
    }

    results = {}  # {指标名: {'L2': [], '提升%': []}}

    for name, scores in metrics.items():
        sorted_idx = np.argsort(scores)[::-1]  # 降序（高风险）
        # sorted_idx = np.argsort(scores) #升序
        l2_list = []
        for ratio in remove_ratios:
            num_remove = int(N_test * ratio)
            keep_idx = sorted_idx[num_remove:]
            if len(keep_idx) == 0:
                l2_list.append(np.nan)
            else:
                u_pred_keep = u_pred_full[keep_idx]
                u_star_keep = u_star_full[keep_idx]
                l2 = np.linalg.norm(u_star_keep - u_pred_keep, 2) / np.linalg.norm(u_star_keep, 2)
                l2_list.append(l2)
        # 计算提升百分比（相对于删除比例为0%时的L2误差）
        l2_0 = l2_list[0]  # 删除0%的L2
        if np.isnan(l2_0) or l2_0 == 0:
            improvement = [np.nan] * len(l2_list)
        else:
            improvement = [(l2_0 - l2) / l2_0 * 100 if not np.isnan(l2) else np.nan for l2 in l2_list]
        results[name] = {'L2': l2_list, '提升%': improvement}

    for name, res in results.items():
        for r, l2, imp in zip(remove_ratios, res['L2'], res['提升%']):
            all_records.append({
                'seed': seed, 'metric': name, 'order': 'desc',
                'remove_ratio': r, 'L2': l2, 'improvement_pct': imp,
                'mean_failure': mean_failure,
            })

    # 打印表格
    try:
        import pandas as pd
        df = pd.DataFrame({'删除比例(%)': (remove_ratios * 100).astype(int)})
        for name in metrics.keys():
            df[f'{name}_L2'] = results[name]['L2']
            df[f'{name}_提升%'] = results[name]['提升%']
        print('='*80)
        print(f'【种子 {seed}】各指标降序排列不同删除比例下的 L2 误差及相比不删除的提升百分比')
        print(df.round(6).to_string(index=False))
    except ImportError:
        # 无 pandas 时简单打印
        print('='*80)
        print(f'【种子 {seed}】各指标降序排列不同删除比例下的 L2 误差')
        print('删除比例%  ' + '  '.join(metrics.keys()))
        for i, ratio in enumerate(remove_ratios):
            row = f'{int(ratio*100):>6}%  '
            for name in metrics.keys():
                row += f'{results[name]["L2"][i]:.6f}  '
            print(row)
        # 打印提升
        print('\n提升百分比 (%)')
        print('删除比例%  ' + '  '.join(metrics.keys()))
        for i, ratio in enumerate(remove_ratios):
            row = f'{int(ratio*100):>6}%  '
            for name in metrics.keys():
                row += f'{results[name]["提升%"][i]:.2f}  '
            print(row)
    print('='*80)


pd.DataFrame(all_records).to_csv('kan_biaozhun_results.csv', index=False)

Start warmup pretraining:
Start Pretrain Loss: 4.7008


0it [00:00, ?it/s]
  0%|          | 0/500 [00:00<?, ?it/s]

100%|██████████| 500/500 [00:17<00:00, 27.83it/s]


Warmup pretraining end.
End Pretrain Loss: 0.0000


0it [00:00, ?it/s]
100%|██████████| 50/50 [01:39<00:00,  1.99s/it]
0it [00:00, ?it/s]
100%|██████████| 50/50 [01:39<00:00,  1.99s/it]
0it [00:00, ?it/s]
100%|██████████| 50/50 [01:56<00:00,  2.33s/it]
0it [00:00, ?it/s]
100%|██████████| 50/50 [01:40<00:00,  2.00s/it]
0it [00:00, ?it/s]
100%|██████████| 50/50 [01:42<00:00,  2.04s/it]
0it [00:00, ?it/s]
100%|██████████| 50/50 [01:04<00:00,  1.28s/it]
0it [00:00, ?it/s]
100%|██████████| 50/50 [00:30<00:00,  1.62it/s]
0it [00:00, ?it/s]
100%|██████████| 50/50 [00:30<00:00,  1.61it/s]
0it [00:00, ?it/s]
100%|██████████| 50/50 [00:31<00:00,  1.61it/s]
0it [00:00, ?it/s]
100%|██████████| 50/50 [00:31<00:00,  1.61it/s]
0it [00:00, ?it/s]
100%|██████████| 50/50 [00:31<00:00,  1.61it/s]
0it [00:00, ?it/s]
100%|██████████| 50/50 [00:31<00:00,  1.59it/s]
0it [00:00, ?it/s]
100%|██████████| 50/50 [00:31<00:00,  1.60it/s]
0it [00:00, ?it/s]
100%|██████████| 50/50 [00:31<00:00,  1.57it/s]
0it [00:00, ?it/s]
100%|██████████| 50/50 [00:31<00:00,  1.60i

...

当前为第9次循环，种子为12345
Error u : 5.406963e-01
MAE: 2.338646e-01
MSE: 9.019978e-02
总failure值 : 1.731794e-02
【种子 12345】各指标升序排列不同删除比例下的 L2 误差及相比不删除的提升百分比
 删除比例(%)  Residual_L2  Residual_提升%  Failure_L2  Failure_提升%
       0     0.540696      0.000000    0.540696     0.000000
       5     0.542612     -0.354238    0.540318     0.069981
      10     0.545840     -0.951268    0.538951     0.322726
      15     0.545679     -0.921480    0.538472     0.411299
      20     0.548567     -1.455627    0.538875     0.336888
      25     0.548233     -1.393888    0.539672     0.189444
      30     0.550378     -1.790624    0.538601     0.387439
      35     0.548119     -1.372747    0.537214     0.644108
      40     0.546111     -1.001399    0.535991     0.870209
      45     0.547805     -1.314785    0.535953     0.877227
      50     0.546371     -1.049564    0.536388     0.796885
      55     0.547360     -1.232522    0.537669     0.559978
      60     0.547453     -1.249664    0.540569     0.023473


0it [00:00, ?it/s]
100%|██████████| 500/500 [00:08<00:00, 56.43it/s]


Warmup pretraining end.
End Pretrain Loss: 0.0000


0it [00:00, ?it/s]
100%|██████████| 50/50 [01:35<00:00,  1.91s/it]
0it [00:00, ?it/s]
100%|██████████| 50/50 [01:36<00:00,  1.94s/it]
0it [00:00, ?it/s]
100%|██████████| 50/50 [01:38<00:00,  1.96s/it]
0it [00:00, ?it/s]
100%|██████████| 50/50 [01:38<00:00,  1.97s/it]
0it [00:00, ?it/s]
100%|██████████| 50/50 [01:39<00:00,  1.98s/it]
0it [00:00, ?it/s]
100%|██████████| 50/50 [01:42<00:00,  2.04s/it]
0it [00:00, ?it/s]
100%|██████████| 50/50 [01:42<00:00,  2.04s/it]
0it [00:00, ?it/s]
100%|██████████| 50/50 [01:42<00:00,  2.05s/it]
0it [00:00, ?it/s]
100%|██████████| 50/50 [01:40<00:00,  2.02s/it]
0it [00:00, ?it/s]
100%|██████████| 50/50 [01:02<00:00,  1.25s/it]
0it [00:00, ?it/s]
100%|██████████| 50/50 [00:30<00:00,  1.61it/s]
0it [00:00, ?it/s]
100%|██████████| 50/50 [00:30<00:00,  1.61it/s]
0it [00:00, ?it/s]
100%|██████████| 50/50 [00:30<00:00,  1.62it/s]
0it [00:00, ?it/s]
100%|██████████| 50/50 [00:30<00:00,  1.62it/s]
0it [00:00, ?it/s]
100%|██████████| 50/50 [00:31<00:00,  1.61i

当前为第10次循环，种子为54321
Error u : 5.434830e-01
MAE: 2.328721e-01
MSE: 9.113194e-02
总failure值 : 2.582651e-02
【种子 54321】各指标升序排列不同删除比例下的 L2 误差及相比不删除的提升百分比
 删除比例(%)  Residual_L2  Residual_提升%  Failure_L2  Failure_提升%
       0     0.543483      0.000000    0.543483     0.000000
       5     0.544601     -0.205775    0.542474     0.185727
      10     0.544961     -0.272029    0.542168     0.242020
      15     0.545192     -0.314495    0.541598     0.346917
      20     0.545013     -0.281475    0.541860     0.298570
      25     0.547955     -0.822902    0.543612    -0.023735
      30     0.551247     -1.428614    0.545692    -0.406498
      35     0.552683     -1.692771    0.547579    -0.753687
      40     0.555007     -2.120367    0.551309    -1.439983
      45     0.555647     -2.238252    0.553723    -1.884146
      50     0.558458     -2.755369    0.558369    -2.738943
      55     0.557328     -2.547479    0.561782    -3.366947
      60     0.558660     -2.792530    0.565915    -4.127542